In [1]:
from pathlib import Path
import re
import pandas as pd

# Root folder containing participant visit folders like: 3002498_irm_t04-3192767
root_dir = Path(r"D:\02-Raw_data-anat\longitudinal_freesurfer_149\reconall_output")

# Regex to parse folder names and extract participant id + visit code (e.g., t04)
folder_pattern = re.compile(r"^(?P<participant>\d+)_irm_(?P<visit>t\d+)-")

# Regex to extract numeric value from the 3 required Measure lines
measure_patterns = {
    "lhSurfaceHoles": re.compile(r"^#\s*Measure\s+lhSurfaceHoles\s*,.*?,\s*([-+]?\d*\.?\d+)\s*,", re.IGNORECASE),
    "rhSurfaceHoles": re.compile(r"^#\s*Measure\s+rhSurfaceHoles\s*,.*?,\s*([-+]?\d*\.?\d+)\s*,", re.IGNORECASE),
    "SurfaceHoles": re.compile(r"^#\s*Measure\s+SurfaceHoles\s*,.*?,\s*([-+]?\d*\.?\d+)\s*,", re.IGNORECASE),
}


def parse_aseg_stats(stats_path: Path):
    """Extract lhSurfaceHoles, rhSurfaceHoles, SurfaceHoles values from one aseg.stats file."""
    values = {k: None for k in measure_patterns.keys()}

    with stats_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            for key, pat in measure_patterns.items():
                if values[key] is None:
                    m = pat.match(line.strip())
                    if m:
                        values[key] = float(m.group(1))

            # Stop early if all 3 values are already found
            if all(v is not None for v in values.values()):
                break

    return values


# Collect rows at the participant-visit level first
long_rows = []
for participant_folder in root_dir.iterdir():
    if not participant_folder.is_dir():
        continue

    m = folder_pattern.match(participant_folder.name)
    if not m:
        continue

    participant_id = m.group("participant")
    visit_code = m.group("visit")
    stats_file = participant_folder / "stats" / "aseg.stats"

    if not stats_file.exists():
        continue

    vals = parse_aseg_stats(stats_file)
    long_rows.append(
        {
            "participant": participant_id,
            "visit": visit_code,
            "lhSurfaceHoles": vals["lhSurfaceHoles"],
            "rhSurfaceHoles": vals["rhSurfaceHoles"],
            "SurfaceHoles": vals["SurfaceHoles"],
        }
    )

if not long_rows:
    raise RuntimeError(f"No valid aseg.stats files found under: {root_dir}")

long_df = pd.DataFrame(long_rows)

# Sort visits numerically: t01, t02, ..., t10
long_df["visit_num"] = long_df["visit"].str.extract(r"t(\d+)").astype(float)
long_df = long_df.sort_values(["participant", "visit_num"])

# Create visit index per participant (1 = first visit, 2 = second visit, ...)
long_df["visit_index"] = long_df.groupby("participant").cumcount() + 1

# Keep first 2 visits for the requested layout (3 columns for visit 1 + 3 columns for visit 2)
wide_df = (
    long_df[long_df["visit_index"].isin([1, 2])]
    .pivot(index="participant", columns="visit_index", values=["lhSurfaceHoles", "rhSurfaceHoles", "SurfaceHoles"])
)

# Flatten multi-index columns into requested order
column_map = {
    ("lhSurfaceHoles", 1): "lh_surface_holes_visit1",
    ("rhSurfaceHoles", 1): "rh_surface_holes_visit1",
    ("SurfaceHoles", 1): "surface_holes_visit1",
    ("lhSurfaceHoles", 2): "lh_surface_holes_visit2",
    ("rhSurfaceHoles", 2): "rh_surface_holes_visit2",
    ("SurfaceHoles", 2): "surface_holes_visit2",
}

for col in column_map:
    if col not in wide_df.columns:
        wide_df[col] = pd.NA

wide_df = wide_df[list(column_map.keys())].rename(columns=column_map).reset_index()
wide_df = wide_df.rename(columns={"participant": "participant_number"})

# Save final CSV to your requested location
output_csv = root_dir / "surface_holes_by_participant_visits.csv"
wide_df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Rows: {len(wide_df)}")
wide_df.head()

Saved: D:\02-Raw_data-anat\longitudinal_freesurfer_149\reconall_output\surface_holes_by_participant_visits.csv
Rows: 75


,participant_number,lhSurfaceHoles,rhSurfaceHoles,SurfaceHoles,lhSurfaceHoles,rhSurfaceHoles,SurfaceHoles
visit_index,,1,1,1,2,2,2
0,3002498,11.0,11.0,22.0,16.0,14.0,30.0
1,3025432,19.0,18.0,37.0,16.0,8.0,24.0
2,3100205,37.0,19.0,56.0,31.0,21.0,52.0
3,3123186,10.0,11.0,21.0,13.0,11.0,24.0
4,3149469,18.0,21.0,39.0,18.0,16.0,34.0
